# Деплой SourceExecutor в Avalanche

Этот ноутбук разворачивает `SourceExecutor.sol` из текущей папки в Avalanche C-Chain.

## Подготовка
- Используй kernel из `myvenv`.
- Экспортируй переменные окружения:
  - `AVALANCHE_RPC_URL`
  - `DEPLOYER_PRIVATE_KEY`
  - `ODOS_ROUTER`
  - `USDC_ADDRESS`
  - `TOKEN_MESSENGER`
- На кошельке деплоера должен быть AVAX для газа.


In [ ]:
# Если пакетов нет, раскомментируй строку ниже:
# !pip install web3 py-solc-x

import os
from pathlib import Path

from solcx import compile_standard, install_solc
from web3 import Web3

CHAIN_ID = 43114


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Missing env var: {name}")
    return value


rpc_url = require_env("AVALANCHE_RPC_URL")
private_key = require_env("DEPLOYER_PRIVATE_KEY")
odos_router = Web3.to_checksum_address(require_env("ODOS_ROUTER"))
usdc_address = Web3.to_checksum_address(require_env("USDC_ADDRESS"))
token_messenger = Web3.to_checksum_address(require_env("TOKEN_MESSENGER"))


In [ ]:
w3 = Web3(Web3.HTTPProvider(rpc_url))
if not w3.is_connected():
    raise ConnectionError("Не удалось подключиться к RPC")

install_solc("0.8.19")
source = Path("SourceExecutor.sol").read_text()

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {"SourceExecutor.sol": {"content": source}},
        "settings": {"outputSelection": {"*": {"*": ["abi", "evm.bytecode"]}}},
    },
    solc_version="0.8.19",
)

contract_data = compiled["contracts"]["SourceExecutor.sol"]["SourceExecutor"]
abi = contract_data["abi"]
bytecode = contract_data["evm"]["bytecode"]["object"]


In [ ]:
account = w3.eth.account.from_key(private_key)
print("Deployer:", account.address)

contract = w3.eth.contract(abi=abi, bytecode=bytecode)

constructor = contract.constructor(odos_router, usdc_address, token_messenger)

gas_estimate = constructor.estimate_gas({"from": account.address})

tx = constructor.build_transaction(
    {
        "from": account.address,
        "nonce": w3.eth.get_transaction_count(account.address),
        "gas": int(gas_estimate * 1.2),
        "gasPrice": w3.eth.gas_price,
        "chainId": CHAIN_ID,
    }
)

signed = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed.rawTransaction)
print("Tx hash:", tx_hash.hex())

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Contract address:", receipt.contractAddress)
